# Performance Tuning V8 — Legal-BERT Fine-Tune (Colab GPU)

**Goal:** Push micro-F1 past 0.85 by fine-tuning Legal-BERT on 512-token chunks and max-pooling to contract level.

**Baseline (from CPU ablation in `performance_tuning.ipynb`):**
- V1 TF-IDF + LR: 0.7936
- V5 (best CPU): 0.8020 (TF-IDF + calibration + per-clause thresholds)
- V6/V7 (MiniLM ensembles): regressed — dense embeddings can't beat TF-IDF with only 408 train contracts

**This notebook:**
1. Mount repo / install deps
2. Load CUAD, filter clauses, chunk with Legal-BERT tokenizer (stride=128)
3. Fine-tune `nlpaueb/legal-bert-base-uncased` for 4 epochs on T4
4. Max-pool chunk probabilities → contract-level scores
5. Tune per-clause thresholds on val, report metrics
6. Save artifacts for the webapp and update the ablation table

**Wall time target:** ~20-25 min on T4, ~8-12 min on A100.

## Section 0 — Colab setup

Uncomment the Colab-only cells when running on Colab. If running locally on a machine with a CUDA GPU, skip them.

In [ ]:
# Colab: install deps (skip locally — handled by requirements.txt)
# !pip install -q "transformers>=4.36,<4.45" "datasets>=2.14" "accelerate>=0.26" scikit-learn joblib huggingface_hub

In [ ]:
# Colab: mount Drive and cd into the project checkout
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/BT5153/crazy-tharp-7c055d   # <-- adjust to your repo path

# Or clone fresh:
# !git clone <your-repo-url> /content/project && cd /content/project

import os, sys
print('cwd:', os.getcwd())
print('python:', sys.version.split()[0])

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: no GPU detected — this notebook will be extremely slow on CPU')

In [ ]:
import random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Section 1 — Load CUAD, filter, chunk

We reuse `prepare_chunked_splits`, which internally:
- builds clause↔id mappings
- splits contracts with seed=42 (same split as V1-V7, so V8 is directly comparable)
- tokenizes each contract into overlapping 512-token chunks (stride=128) with the **Legal-BERT tokenizer**
- propagates the contract's multi-label vector to every chunk that overlaps a labelled span

In [ ]:
from data_loading import load_cuad
from preprocessing import filter_clauses, prepare_chunked_splits

cuad_df = load_cuad('data/cuad')
cuad_filtered, excluded = filter_clauses(cuad_df, min_positives=20)
print(f'Clauses kept: {cuad_filtered["clause_type"].nunique()} | excluded: {len(excluded)}')

In [ ]:
MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'

splits = prepare_chunked_splits(
    cuad_filtered,
    model_name=MODEL_NAME,
    max_length=512,
    stride=128,
    seed=SEED,
)

print(f"Contracts — train: {len(splits['train_records'])}, val: {len(splits['val_records'])}, test: {len(splits['test_records'])}")
print(f"Chunks    — train: {len(splits['train_examples'])}, val: {len(splits['val_examples'])}, test: {len(splits['test_examples'])}")
print(f"Labels    — {len(splits['id_to_clause'])}")

## Section 2 — Fine-tune Legal-BERT

Uses the existing `train_legal_bert_cuad` helper, which wraps `train_bert_cuad` and runs the standard CUAD recipe:
- AdamW, lr=2e-5, weight_decay=0.01, linear warmup (10%)
- BCEWithLogitsLoss with per-label `pos_weight` (from `compute_pos_weight`)
- Per-sample downweighting of all-negative chunks (from `compute_sample_weights`)
- Validation logits/labels collected at the end of each epoch; best-epoch model kept

If T4 OOMs at `batch_size=16`, drop to 8 (slower but fits easily).

In [ ]:
from training import train_legal_bert_cuad, choose_device

device = choose_device()
print('Training on:', device)

artifacts = train_legal_bert_cuad(
    train_dataset=splits['train_dataset'],
    val_dataset=splits['val_dataset'],
    train_examples=splits['train_examples'],
    tokenizer=splits['tokenizer'],
    id_to_clause=splits['id_to_clause'],
    model_name=MODEL_NAME,
    epochs=4,
    batch_size=16,        # T4 fits 16 at max_length=512; drop to 8 if OOM
    learning_rate=2e-5,
    device=device,
)

print('\n=== Chunk-level val metrics (global threshold) ===')
print(artifacts.val_metrics)
print('best global threshold:', artifacts.best_threshold)

In [ ]:
# Sanity check: training curve should climb monotonically
import matplotlib.pyplot as plt

hist = artifacts.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if 'train_loss' in hist.columns:
    ax[0].plot(hist['epoch'], hist['train_loss'], marker='o', label='train')
if 'val_loss' in hist.columns:
    ax[0].plot(hist['epoch'], hist['val_loss'],   marker='o', label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
if 'val_micro_f1' in hist.columns:
    ax[1].plot(hist['epoch'], hist['val_micro_f1'], marker='o', color='C2')
ax[1].set_title('Val chunk-level micro-F1'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()

## Section 3 — Max-pool chunks → contract-level probabilities

Chunk-level F1 is optimistic: every chunk has the contract's label, so a model can get decent F1 by learning "if the contract contains this clause, this chunk contains it too". What matters for the webapp is **contract-level** F1: does the contract contain the clause at all?

Max-pool gives us that: if *any* chunk scores high for a clause, the contract is flagged. Uses the existing `aggregate_contract_predictions` helper.

In [ ]:
import numpy as np, pandas as pd
from preprocessing import aggregate_contract_predictions

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

id_to_clause = splits['id_to_clause']
clause_to_id = splits['clause_to_id']
val_examples = splits['val_examples']
val_records  = splits['val_records']

val_probs = _sigmoid(artifacts.val_logits)           # [n_chunks, n_labels]
print('val_probs shape:', val_probs.shape)

# Build long-format chunk DataFrame
rows = []
for chunk_idx, ex in enumerate(val_examples):
    for clause_id in range(val_probs.shape[1]):
        rows.append({
            'contract_title': ex['contract_title'],
            'clause_type':    id_to_clause[clause_id],
            'score':          float(val_probs[chunk_idx, clause_id]),
            'chunk_index':    ex['chunk_index'],
        })
chunk_long_df = pd.DataFrame(rows)
print('chunk_long_df:', chunk_long_df.shape)

# Max-pool per (contract, clause)
contract_preds = aggregate_contract_predictions(chunk_long_df)
print('contract_preds:', contract_preds.shape)

In [ ]:
# Pivot to [n_contracts, n_labels] probability matrix + build labels
contract_titles = [r['contract_title'] for r in val_records]
clause_cols     = [id_to_clause[i] for i in range(len(id_to_clause))]

pivot = (
    contract_preds
    .pivot(index='contract_title', columns='clause_type', values='max_score')
    .reindex(index=contract_titles, columns=clause_cols)
    .fillna(0.0)
)
contract_probs  = pivot.values
contract_logits = np.log(np.clip(contract_probs, 1e-7, 1-1e-7) / np.clip(1 - contract_probs, 1e-7, 1-1e-7))

contract_labels = np.zeros((len(val_records), len(id_to_clause)), dtype=int)
for i, rec in enumerate(val_records):
    for clause_name in rec['clause_spans']:
        if clause_name in clause_to_id:
            contract_labels[i, clause_to_id[clause_name]] = 1

print('contract_probs:', contract_probs.shape)
print('contract_labels:', contract_labels.shape, 'positives:', contract_labels.sum())

## Section 4 — Per-clause thresholds + final metrics

In [ ]:
from evaluation import tune_per_clause_thresholds, compute_aggregate_metrics, compute_per_clause_metrics

# Per-clause thresholds tuned on val (for V1-V7 parity)
per_clause_thr = tune_per_clause_thresholds(contract_logits, contract_labels, id_to_clause)

metrics_v8 = compute_aggregate_metrics(
    contract_logits, contract_labels,
    thresholds=per_clause_thr,
    id_to_clause=id_to_clause,
)
print('=== V8 contract-level metrics (per-clause thresholds) ===')
for k, v in metrics_v8.items():
    print(f'  {k:18s} {v:.4f}')

# Also try a single global threshold for comparison
thresh_grid = np.arange(0.05, 0.96, 0.05)
best_global_f1, best_global_t = -1.0, 0.5
from sklearn.metrics import f1_score
for t in thresh_grid:
    preds = (contract_probs >= t).astype(int)
    f1 = f1_score(contract_labels, preds, average='micro', zero_division=0)
    if f1 > best_global_f1:
        best_global_f1, best_global_t = float(f1), float(t)
print(f'\nBest global-threshold micro-F1: {best_global_f1:.4f} at t={best_global_t:.2f}')

In [ ]:
# Per-clause breakdown
# n_positive_train needed for the per-clause metrics helper
train_labels = np.zeros((len(splits['train_records']), len(id_to_clause)), dtype=int)
for i, rec in enumerate(splits['train_records']):
    for clause_name in rec['clause_spans']:
        if clause_name in clause_to_id:
            train_labels[i, clause_to_id[clause_name]] = 1
n_positive_train = {i: int(train_labels[:, i].sum()) for i in range(len(id_to_clause))}

per_clause_df = compute_per_clause_metrics(
    contract_logits, contract_labels,
    thresholds=per_clause_thr,
    id_to_clause=id_to_clause,
    n_positive_train=n_positive_train,
)
per_clause_df.head(10)

## Section 5 — Save artifacts, update ablation table, update chart

In [ ]:
import joblib, os
from pathlib import Path

Path('checkpoints').mkdir(exist_ok=True)
ckpt_path = 'checkpoints/legal_bert_v8_artifacts.joblib'

payload = {
    'variant':          'V8 — Legal-BERT fine-tune',
    'model_state_dict': {k: v.detach().cpu() for k, v in artifacts.model.state_dict().items()},
    'tokenizer_name':   MODEL_NAME,
    'num_labels':       len(id_to_clause),
    'id_to_clause':     id_to_clause,
    'clause_to_id':     clause_to_id,
    'best_threshold':   per_clause_thr,     # dict {clause_name: threshold} — webapp-compatible
    'max_length':       512,
    'stride':           128,
    'val_metrics':      metrics_v8,
}
joblib.dump(payload, ckpt_path, compress=3)
print(f'Saved {ckpt_path} ({os.path.getsize(ckpt_path)/1e6:.1f} MB)')

In [ ]:
# Append V8 to the existing ablation table
Path('figures').mkdir(exist_ok=True)
ablation_path = 'figures/ablation_table.csv'

v1_f1 = 0.7936  # frozen baseline
v8_row = {
    'Variant':         'V8 — Legal-BERT fine-tune',
    'Micro-F1':        round(metrics_v8['micro_f1'], 4),
    'Macro-F1':        round(metrics_v8['macro_f1'], 4),
    'Micro-Precision': round(metrics_v8['micro_precision'], 4),
    'Micro-Recall':    round(metrics_v8['micro_recall'], 4),
    'Δ vs baseline':   round(metrics_v8['micro_f1'] - v1_f1, 4),
}

if os.path.exists(ablation_path):
    df = pd.read_csv(ablation_path)
    df = df[df['Variant'] != v8_row['Variant']]           # dedupe if rerun
    df = pd.concat([df, pd.DataFrame([v8_row])], ignore_index=True)
else:
    df = pd.DataFrame([v8_row])

df.to_csv(ablation_path, index=False)
df

In [ ]:
# Per-clause F1 chart: V1 vs V5 vs V8 (if V1/V5 per-clause files exist; otherwise V8 only)
import matplotlib.pyplot as plt

per_clause_v8 = per_clause_df[['clause_type', 'f1']].rename(columns={'f1': 'V8'})

# Optional: merge V1 + V5 per-clause F1 if they were persisted by performance_tuning.ipynb
joined = per_clause_v8.set_index('clause_type')
for prior in ('figures/per_clause_v1.csv', 'figures/per_clause_v5.csv'):
    if os.path.exists(prior):
        tag = Path(prior).stem.split('_')[-1].upper()
        d = pd.read_csv(prior)[['clause_type', 'f1']].rename(columns={'f1': tag}).set_index('clause_type')
        joined = joined.join(d, how='outer')
joined = joined.fillna(0.0).sort_values('V8', ascending=True)

fig, ax = plt.subplots(figsize=(14, 10))
joined.plot(kind='barh', ax=ax, colormap='tab10')
ax.set_xlabel('F1'); ax.set_title('Per-Clause F1 — V8 (Legal-BERT) vs prior variants')
ax.axvline(x=0.8, color='red', linestyle='--', alpha=0.4, label='F1 = 0.8')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('figures/per_clause_f1_chart_v8.png', dpi=150, bbox_inches='tight')
plt.show()
per_clause_v8.to_csv('figures/per_clause_v8.csv', index=False)

## Section 6 — Download artifacts back to local machine

On Colab, download the checkpoint + figures to your laptop so you can swap it into the webapp.

In [ ]:
# Colab-only:
# from google.colab import files
# files.download('checkpoints/legal_bert_v8_artifacts.joblib')
# files.download('figures/ablation_table.csv')
# files.download('figures/per_clause_f1_chart_v8.png')
print('Done. Next steps:')
print('  1. Download legal_bert_v8_artifacts.joblib to your Mac')
print('  2. Back up: cp checkpoints/tfidf_lr_artifacts.joblib checkpoints/tfidf_lr_artifacts.V5.backup.joblib')
print('  3. Swap:    cp checkpoints/legal_bert_v8_artifacts.joblib checkpoints/tfidf_lr_artifacts.joblib')
print('  4. Run:     streamlit run webapp/app.py')